# Agent Response (Phase 6)

This notebook implements the **Response agent**.

Phase 6 scope:
- Subscribe to Observer city snapshots and Control health updates
- Maintain authoritative status counts
- Publish `${base_topic}/pandemic/response/metrics` once per simulation step

In [ ]:
# Cell purpose: Import modules, load config, and define Phase 6 response topics.
from __future__ import annotations

import json

import simulated_city.mqtt as mqtt
from simulated_city.config import load_config
from simulated_city.epidemic import build_response_metrics

cfg = load_config()
sim_cfg = cfg.simulation
mqtt_cfg = cfg.mqtt

if sim_cfg is None:
    raise ValueError("Phase 6 requires a 'simulation' section in config.yaml")

base_topic = mqtt_cfg.base_topic
topic_city_snapshot = f"{base_topic}/pandemic/observer/city_snapshot"
topic_health_update = f"{base_topic}/pandemic/control/health_update"
topic_metrics = f"{base_topic}/pandemic/response/metrics"

print(
    f"Response topics: subscribe=[{topic_city_snapshot}, {topic_health_update}], "
    f"publish={topic_metrics}"
)

In [ ]:
# Cell purpose: Define response state, helper functions, and per-step metric publishing.
counts_by_status = {"susceptible": 0, "infected": 0, "recovered": 0}
person_status: dict[str, str] = {}
snapshot_total_population = sim_cfg.population_size
latest_ts_by_step: dict[int, str] = {}
new_infections_by_step: dict[int, int] = {}
new_recoveries_by_step: dict[int, int] = {}
published_steps: set[int] = set()
metrics_published = 0
health_updates_seen = 0
snapshots_seen = 0
malformed_messages = 0
last_step_seen = -1
client = None


def apply_status_change(person_id: str, new_status: str) -> None:
    old_status = person_status.get(person_id)
    if old_status == new_status:
        return

    if old_status in counts_by_status:
        counts_by_status[old_status] = max(0, counts_by_status[old_status] - 1)

    person_status[person_id] = new_status
    counts_by_status.setdefault(new_status, 0)
    counts_by_status[new_status] += 1


def maybe_publish_step_metrics(step: int) -> None:
    global metrics_published
    if step in published_steps:
        return

    ts = latest_ts_by_step.get(step, "")
    payload = build_response_metrics(
        step=step,
        ts=ts,
        susceptible=counts_by_status.get("susceptible", 0),
        infected=counts_by_status.get("infected", 0),
        recovered=counts_by_status.get("recovered", 0),
        new_infections=new_infections_by_step.get(step, 0),
        new_recoveries=new_recoveries_by_step.get(step, 0),
    )
    mqtt.publish_json_checked(client, topic_metrics, payload)
    published_steps.add(step)
    metrics_published += 1

In [ ]:
# Cell purpose: Connect response MQTT client, process Observer+Control messages, and publish step metrics.
client = mqtt.connect_mqtt(mqtt_cfg, client_id_suffix="response")
print(f"Connected to MQTT broker at {mqtt_cfg.host}:{mqtt_cfg.port}")


def on_message(client_obj, userdata, msg):
    global snapshots_seen, health_updates_seen, malformed_messages, last_step_seen, snapshot_total_population

    try:
        payload = json.loads(msg.payload.decode())
    except Exception:
        malformed_messages += 1
        return

    if msg.topic == topic_city_snapshot:
        step = payload.get("step")
        ts = payload.get("ts", "")
        population = payload.get("population")

        if not isinstance(step, int):
            malformed_messages += 1
            return
        if not isinstance(population, int):
            malformed_messages += 1
            return

        snapshot_total_population = population
        latest_ts_by_step[step] = ts
        snapshots_seen += 1
        last_step_seen = max(last_step_seen, step)

        maybe_publish_step_metrics(step)
        return

    if msg.topic == topic_health_update:
        step = payload.get("step")
        ts = payload.get("ts", "")
        person_id = payload.get("person_id")
        to_status = payload.get("to_status")

        if not isinstance(step, int):
            malformed_messages += 1
            return
        if not isinstance(person_id, str):
            malformed_messages += 1
            return
        if to_status not in ("susceptible", "infected", "recovered"):
            malformed_messages += 1
            return

        apply_status_change(person_id, to_status)
        latest_ts_by_step[step] = ts
        if to_status == "infected":
            new_infections_by_step[step] = new_infections_by_step.get(step, 0) + 1
        elif to_status == "recovered":
            new_recoveries_by_step[step] = new_recoveries_by_step.get(step, 0) + 1
        health_updates_seen += 1
        last_step_seen = max(last_step_seen, step)


client.on_message = on_message
client.subscribe(topic_city_snapshot)
client.subscribe(topic_health_update)
print(f"Subscribed to {topic_city_snapshot}")
print(f"Subscribed to {topic_health_update}")
print("Response is now listening. Metrics are published once per city snapshot step.")

In [ ]:
# Cell purpose: Print response summary and disconnect cleanly when finished.
print(
    f"Response summary: snapshots_seen={snapshots_seen}, health_updates_seen={health_updates_seen}, "
    f"metrics_published={metrics_published}, malformed_messages={malformed_messages}, "
    f"last_step_seen={last_step_seen}, counts={counts_by_status}"
)

connector = getattr(client, "_simulated_city_connector", None)
if connector is not None:
    connector.disconnect()
    print("Disconnected from MQTT broker.")
else:
    print("No connector reference found; client disconnect skipped.")